In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

RECORD_ID = "18342253"
API_URL = f"https://zenodo.org/api/records/{RECORD_ID}"

# 1. Consulta REST a la API de Zenodo para obtener metadatos y enlaces de descarga
response = requests.get(API_URL, timeout=30)
response.raise_for_status()
data_json = response.json()

doi = data_json.get("doi", f"10.5281/zenodo.{RECORD_ID}")

# Identificar dinámicamente el enlace de descarga del archivo CSV desde la API
download_url = None
for file_info in data_json.get("files", []):
    if file_info.get("key", "").endswith(".csv"):
        download_url = file_info.get("links", {}).get("self") or file_info.get("links", {}).get("content")
        break

if not download_url:
    download_url = f"https://zenodo.org/records/{RECORD_ID}/files/dataRSM.csv?download=1"

# 2. Descarga del archivo usando la referencia obtenida vía API
file_resp = requests.get(download_url, timeout=60)
file_resp.raise_for_status()

local_path = "/tmp/dataRSM.csv"
with open(local_path, "wb") as f:
    f.write(file_resp.content)

# 3. Carga e ingesta a la Capa Bronze
pdf = pd.read_csv(local_path, sep=";", decimal=",")
ext_df = (
    spark.createDataFrame(pdf)
    .withColumn("_source", F.lit("zenodo_rest_api"))
    .withColumn("_doi", F.lit(doi))
    .withColumn("_ingested_at", F.current_timestamp())
)

ext_df.write.mode("overwrite").saveAsTable(T_EXT_RAW)
print("Bronze market_reference_raw:", spark.table(T_EXT_RAW).count(), "filas")
display(spark.table(T_EXT_RAW).limit(10))